### Imports and Configuration

In [112]:
import os
import numpy as np
import pandas as pd
import warnings
from scipy.stats import linregress
import matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings('ignore')


### Paths

In [113]:
batch_sen = pd.read_csv("../data/rawdata/uhn_breast/batch_metrics.csv")
cnv = pd.read_csv("../data/rawdata/uhn_breast/omics/cnv_hugo.tsv", sep="\t", index_col=0)
mut = pd.read_csv("../data/rawdata/uhn_breast/omics/mutation_hugo.tsv", sep="\t", index_col=0)

# batch name contains CARBOPLATIN but not "-"
car = batch_sen[batch_sen["batch.name"].str.contains("CARBOPLATIN") & ~batch_sen["batch.name"].str.contains("-") & ~batch_sen["batch.name"].str.contains("RES")].reset_index(drop=True)

car["model.id"] = car["batch.name"].str.split(".").str[0]
car["drug.id"] = car["batch.name"].str.split(".").str[1]

mask = car["model.id"].str.match(r'^\d', na=False)
car.loc[mask, "model.id"] = "S" + car.loc[mask, "model.id"]

# overlap with cnv/mutation data
car = car[car["model.id"].isin(cnv.columns)].reset_index(drop=True)
car = car.sort_values("model.id").reset_index(drop=True)

#car = car[["model.id", "angle", "mRECIST", "TGI"]].set_index("model.id")
# car = car.set_index("model.id")
car.to_csv("../data/procdata/3_carboplatin/UHN_car_sensitivity.csv")
car["model.id"].to_csv("../data/procdata/3_carboplatin/UHN_nest_input/complete_cohort/cell2ind.txt", header=False, sep = "\t")
print(f"Total CARBOPLATIN models: {len(car)}")
print(car["mRECIST"].value_counts())

car_strict = car[car["n_replicates"] >= 2].reset_index(drop=True)
car_strict["model.id"].to_csv("../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/cell2ind.txt", header=False, sep = "\t")
print(f"\nSTRICT cohort size: {len(car_strict)}")
print(car_strict["mRECIST"].value_counts())

Total CARBOPLATIN models: 67
mRECIST
SD    30
PD    26
PR     7
CR     4
Name: count, dtype: int64

STRICT cohort size: 37
mRECIST
SD    17
PD    14
PR     3
CR     3
Name: count, dtype: int64


### Prepare CNV

In [114]:
cnv.head()

,S108099,REF020,S45086,S71208,S100534,S53782,REF022,REF032,REF061,REF048,...,S100804,REF043.2,REF043.3,REF063,S28077,REF057,REF030,S44441,S69196,S90189
hugo.id,,,,,,,,,,,,,,,,,,,,,
A1BG,4.0,4.0,1.0,3.0,3.0,1.0,1.0,3.0,4.0,1.0,...,NaN,4.0,5.0,NaN,3.0,NaN,3.0,NaN,1.0,1.0
A1BG-AS1,4.0,4.0,1.0,3.0,3.0,1.0,1.0,3.0,4.0,1.0,...,NaN,4.0,5.0,NaN,3.0,NaN,3.0,NaN,1.0,1.0
A1CF,3.0,NaN,6.0,4.0,NaN,3.0,NaN,4.0,NaN,NaN,...,NaN,0.0,0.0,NaN,5.0,NaN,3.0,3.0,NaN,3.0
A2M,4.0,3.0,3.0,NaN,5.0,3.0,3.0,1.0,NaN,NaN,...,NaN,0.0,1.0,3.0,5.0,NaN,5.0,NaN,3.0,1.0
A2M-AS1,4.0,3.0,3.0,NaN,5.0,3.0,3.0,1.0,NaN,NaN,...,NaN,0.0,1.0,3.0,5.0,NaN,5.0,NaN,3.0,1.0


In [115]:
# Define the explicit functional terms (the whitelist)
functional_whitelist = {
    "Missense_Mutation",
    "Nonsense_Mutation",
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Splice_Site",
    "Translation_Start_Site",
    "Nonstop_Mutation"
}

def binarize_mutation_inclusive(val):
    """
    Binarizes mutation data by checking for the presence of 
    explicitly functional variant classifications.
    """
    if pd.isna(val) or val == "" or str(val).lower() == "nan":
        return 0
    
    # Split by comma for cases where multiple mutations are collapsed into one cell
    # Strip whitespace and ensure we handle potential case sensitivity issues
    parts = {p.strip() for p in str(val).split(',')}
    
    # INCLUSIVE LOGIC: If ANY of the parts are in the whitelist, it's a 1
    if any(p in functional_whitelist for p in parts):
        return 1
    
    # Otherwise (e.g., only "Silent", "Intron", or unknown terms), return 0
    return 0

In [116]:
# Load in gene file
gene_file = "../data/rawdata/nest_vnn/gene2ind.txt"

# Load in nesv_vnn genes
gene_map = pd.read_csv(gene_file, sep="\t", header=None)
gene_map.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/complete_cohort/gene2ind.txt", sep="\t", header=False, index=False)
gene_map.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/gene2ind.txt", sep="\t", header=False, index=False)

gene_map.columns = ["index", "gene"]
gene_list = gene_map["gene"].tolist()
print("NEST_VNN Gene:", len(gene_list))

# Prepare mutation matrix
#mut_bin = mut.map(lambda x: 0 if (pd.isna(x) or x == "0" or str(x).lower() == "silent") else 1)
mut_bin = mut.map(binarize_mutation_inclusive)

mut_complete = mut_bin.loc[mut_bin.index.isin(gene_list), mut_bin.columns.isin(car["model.id"])]
print(f"Complete shape: {mut_complete.shape}")
mut_strict = mut_bin.loc[mut_bin.index.isin(gene_list), mut_bin.columns.isin(car_strict["model.id"])]
print(f"Strict shape: {mut_strict.shape}")

# Prepare CNV matrix
cnv_complete = cnv.loc[cnv.index.isin(gene_list), cnv.columns.isin(car["model.id"])]
print(f"Complete shape: {cnv_complete.shape}")
cnv_strict = cnv.loc[cnv.index.isin(gene_list), cnv.columns.isin(car_strict["model.id"])]
print(f"Strict shape: {cnv_strict.shape}")




NEST_VNN Gene: 718
Complete shape: (670, 67)
Strict shape: (670, 37)
Complete shape: (711, 67)
Strict shape: (711, 37)


In [117]:
gene_list 

['ABCB1',
 'ABCC3',
 'ABL1',
 'ABL2',
 'ACTA2',
 'ACVR1B',
 'ADGRA2',
 'AJUBA',
 'AKT1',
 'AKT2',
 'AKT3',
 'ALK',
 'ALOX12B',
 'AMER1',
 'ANKRD11',
 'APC',
 'APOB',
 'AR',
 'ARAF',
 'ARFRP1',
 'ARHGAP26',
 'ARHGAP35',
 'ARID1A',
 'ARID1B',
 'ARID2',
 'ARID5B',
 'ASNS',
 'ASXL1',
 'ASXL2',
 'ATIC',
 'ATM',
 'ATP7B',
 'ATR',
 'ATRX',
 'AURKA',
 'AURKB',
 'AXIN1',
 'AXIN2',
 'AXL',
 'B2M',
 'BAP1',
 'BARD1',
 'BCAR1',
 'BCL10',
 'BCL11B',
 'BCL2',
 'BCL2L1',
 'BCL2L11',
 'BCL2L12',
 'BCL2L2',
 'BCL6',
 'BCL7A',
 'BCLAF1',
 'BCOR',
 'BCORL1',
 'BCR',
 'BIRC3',
 'BLM',
 'BMPR1A',
 'BRAF',
 'BRCA1',
 'BRCA2',
 'BRD4',
 'BRIP1',
 'BTG1',
 'BTK',
 'BUB1B',
 'C11orf65',
 'C3orf70',
 'C8orf34',
 'CADM2',
 'CALR',
 'CARD11',
 'CASP8',
 'CASR',
 'CBFB',
 'CBL',
 'CBLB',
 'CBLC',
 'CBR3',
 'CCDC6',
 'CCND1',
 'CCND2',
 'CCND3',
 'CCNE1',
 'CCNE2',
 'CD19',
 'CD22',
 'CD274',
 'CD276',
 'CD40',
 'CD70',
 'CD79A',
 'CD79B',
 'CDC25A',
 'CDC73',
 'CDH1',
 'CDH2',
 'CDK12',
 'CDK4',
 'CDK5',
 'CDK6',


### Complete dataset

In [118]:
print(cnv_strict.loc['ABL2', 'REF010'])
cna = (cnv_strict >= 7).astype(int)
cna.loc['ABL2', 'REF010']

6.0


np.int64(0)

In [104]:
cna.loc['ABL2', 'REF010']

np.int64(0)

In [119]:
# Prepare CNV matrices
cnv_complete.loc['ABL2', 'REF010']
cna = (cnv_complete >= 7).astype(int)
cnd = (cnv_complete == 0).astype(int)
cna = cna.reindex(gene_list).fillna(0)
cnd = cnd.reindex(gene_list).fillna(0)
mut_complete = mut_complete.reindex(gene_list).fillna(0)

# transpose
mut_complete = mut_complete.T
cna = cna.T
cnd = cnd.T

# convert to int
mut_complete = mut_complete.astype(int)
cna = cna.astype(int)
cnd = cnd.astype(int)

# 
mut_complete = mut_complete.reindex(car["model.id"])
cna = cna.reindex(car["model.id"])
cnd = cnd.reindex(car["model.id"])

# Save NeST input files
mut_complete.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/complete_cohort/cell2mutation.txt", header=False, index=False)
cna.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/complete_cohort/cell2amplification.txt", header=False, index=False)
cnd.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/complete_cohort/cell2cndeletion.txt", header=False, index=False)

In [120]:
print(mut_complete.shape)
print(cna.shape)
print(cnd.shape)

(67, 718)
(67, 718)
(67, 718)


In [121]:
test_df = pd.DataFrame({
    0: car["model.id"],
    1: ["CARBOPLATIN"] * len(car["model.id"])
})
test_df["AUC"] = 0
test_df["dataset"] = "PDX"

test_path = "../data/procdata/3_carboplatin/UHN_nest_input/complete_cohort/test_data.txt"
test_df.to_csv(test_path, sep="\t", header=False, index=False)

### Strict dataset 

In [122]:
# Prepare CNV matrices
cna = (cnv_strict >= 7).astype(int)
cnd = (cnv_strict == 0).astype(int)
cna = cna.reindex(gene_list).fillna(0)
cnd = cnd.reindex(gene_list).fillna(0)
mut_strict = mut_strict.reindex(gene_list).fillna(0)

# transpose
mut_strict = mut_strict.T
cna = cna.T
cnd = cnd.T

# convert to int
mut_strict = mut_strict.astype(int)
cna = cna.astype(int)
cnd = cnd.astype(int)

mut_strict = mut_strict.reindex(car_strict["model.id"])
cna = cna.reindex(car_strict["model.id"])
cnd = cnd.reindex(car_strict["model.id"])

# Save NeST input files
mut_strict.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/cell2mutation.txt", header=False, index=False)
#mut_strict.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/cell2mutation_test.txt", header=False)
cna.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/cell2amplification.txt", header=False, index=False)
#cna.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/cell2amplification.txt")
cnd.to_csv("../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/cell2cndeletion.txt", header=False, index=False)

In [123]:
print(mut_strict.shape)
print(cna.shape)
print(cnd.shape)

(37, 718)
(37, 718)
(37, 718)


In [124]:
test_df = pd.DataFrame({
    0: car_strict["model.id"],
    1: ["CARBOPLATIN"] * len(car_strict["model.id"])
})
test_df["AUC"] = 0
test_df["dataset"] = "PDX"

test_path = "../data/procdata/3_carboplatin/UHN_nest_input/strict_cohort/test_data.txt"
test_df.to_csv(test_path, sep="\t", header=False, index=False)